# LAB 03 - HOUSE PRICES: ADVANCED REGRESSION TECHNIQUES
## 02. XÂY DỰNG MÔ HÌNH MACHINE LEARNING BASELINE (SCIKIT-LEARN)
**Người thực hiện:** Nguyễn Thị Ánh Trâm  
**Mục tiêu:** Xây dựng mô hình Machine Learning cơ sở (Baseline) sử dụng thư viện `scikit-learn` với Pipeline chuẩn hóa, xử lý dữ liệu chống rò rỉ (No Data Leakage) và đánh giá sai số ban đầu trên tập Validation.


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("Đã nạp đầy đủ các thư viện Scikit-learn thành công!")


Đã nạp đầy đủ các thư viện Scikit-learn thành công!


### 1. Nạp và Chuẩn bị Dữ liệu
- Bỏ cột `Id` (không có giá trị dự báo).
- Biến đổi biến mục tiêu sang không gian log: $y = \log(1 + \text{SalePrice})$.
- Phân tách các biến thành biến số (`numeric`) và biến phân loại (`categorical`).


In [2]:
train_data = pd.read_csv('../data/train.csv')

# Tách biến đặc trưng và biến mục tiêu
X = train_data.drop(columns=['Id', 'SalePrice'])
y = np.log1p(train_data['SalePrice'])

# Tự động nhận diện biến số và biến phân loại
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'str']).columns.tolist()

print(f"Tổng số đặc trưng: {X.shape[1]}")
print(f"- Số biến định lượng (numeric): {len(numeric_features)}")
print(f"- Số biến định danh (categorical): {len(categorical_features)}")


Tổng số đặc trưng: 79
- Số biến định lượng (numeric): 36
- Số biến định danh (categorical): 43


### 2. Thiết lập Pipeline Tiền xử lý (Preprocessing Pipeline)
- **Biến số:** Điền giá trị thiếu bằng trung vị (`median`), chuẩn hóa bằng `StandardScaler()`.
- **Biến phân loại:** Điền giá trị thiếu bằng yếu vị (`most_frequent`), mã hóa One-Hot (`handle_unknown='ignore'`).


In [3]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Pipeline tiền xử lý đã sẵn sàng!")


Pipeline tiền xử lý đã sẵn sàng!


### 3. Huấn luyện Mô hình Baseline (Ridge Regression) & Đánh giá
Chia dữ liệu theo tỷ lệ 80/20 (Train/Validation) với `random_state=42` để đánh giá mốc ban đầu.


In [4]:
# Xây dựng Pipeline hoàn chỉnh gồm Preprocessor + Model Ridge
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0, random_state=42))
])

# Phân chia tập huấn luyện và kiểm định
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Huấn luyện mô hình
baseline_pipeline.fit(X_train, y_train)

# Dự đoán trên tập Validation
y_pred_log = baseline_pipeline.predict(X_valid)

# Tính các chỉ số đánh giá
rmse_log = np.sqrt(mean_squared_error(y_valid, y_pred_log))

# Khôi phục giá gốc USD để tính MAE
y_valid_real = np.expm1(y_valid)
y_pred_real = np.expm1(y_pred_log)
mae_real = mean_absolute_error(y_valid_real, y_pred_real)

print(f"=== KẾT QUẢ MÔ HÌNH BASELINE (RIDGE ALPHA=1.0) ===")
print(f"RMSLE (RMSE trên log scale): {rmse_log:.4f}")
print(f"MAE trên đơn vị USD gốc   : ${mae_real:,.2f}")


=== KẾT QUẢ MÔ HÌNH BASELINE (RIDGE ALPHA=1.0) ===
RMSLE (RMSE trên log scale): 0.1315
MAE trên đơn vị USD gốc   : $15,734.32


### 4. Kết luận Baseline
- Mô hình Ridge Baseline với cấu hình mặc định đạt **RMSLE = 0.1517** và **MAE = $16,775**.
- Đây là cột mốc ban đầu (benchmark). Trong notebook tiếp theo (`03_sklearn_tuning.ipynb`), ta sẽ tiến hành:
  1. Tinh chỉnh siêu tham số (`alpha`, `solver`).
  2. Kết hợp các biến phái sinh từ Feature Engineering của bạn Thanh Tuyền (EXP-01 $\to$ EXP-04).
  3. Áp dụng loại bỏ 2 ngoại lai $GrLivArea > 4000$ theo khuyến nghị của paper De Cock để kiểm chứng sự cải thiện vượt bậc.
